In [13]:
import mlflow

In [3]:
print(mlflow.__version__)

3.17.2


In [6]:
!python preprocess_data.py --raw_data_path ./data --dest_path ./output

In [12]:
!bash -c "ls output | wc -l"

4


In [43]:
!python train.py

2026/11/01 22:04:44 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/11/01 22:04:48 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


In [52]:
from mlflow.tracking import MlflowClient

client = MlflowClient(tracking_uri="sqlite:///mlflow.db")
exp = client.search_experiments()

print("Experiment :")
for e in exp : 
    print(e.experiment_id, e.name)

Experiment :
0 Default


In [40]:
#delete les runs existantes !!!!!
runs = client.search_runs(["0"])
for r in runs:
    client.delete_run(r.info.run_id)

In [50]:
runs = client.search_runs(experiment_ids=[0])
for r in runs:
    print(r.info.run_id, r.data.params, r.data.metrics)
    print(f"Min sample split : {r.data.params["min_samples_split"]}")

c4257508acf3475ca89972f9fa6a0d69 {'bootstrap': 'True', 'ccp_alpha': '0.0', 'criterion': 'squared_error', 'max_depth': '10', 'max_features': '1.0', 'max_leaf_nodes': 'None', 'max_samples': 'None', 'min_impurity_decrease': '0.0', 'min_samples_leaf': '1', 'min_samples_split': '2', 'min_weight_fraction_leaf': '0.0', 'monotonic_cst': 'None', 'n_estimators': '100', 'n_jobs': 'None', 'oob_score': 'False', 'random_state': '0', 'verbose': '0', 'warm_start': 'False'} {'training_mean_squared_error': 27.083103049009832, 'training_mean_absolute_error': 3.4244715589842163, 'training_r2_score': 0.6673977812886405, 'training_root_mean_squared_error': 5.204142873616157, 'training_score': 0.6673977812886405, 'rmse': 5.431151718191326}
Min sample split : 2


In [ ]:
#command to lauch the mlflow server

#mlflow server --backend-store-uri sqlite:///mlflow.db --default-artifact-root ./artifacts

In [54]:
!python hpo.py

2026/11/01 22:23:41 INFO mlflow.tracking.fluent: Experiment with name 'random-forest-hyperopt' does not exist. Creating a new experiment.
🏃 View run nosy-bug-773 at: http://127.0.0.1:5000/#/experiments/1/runs/919eff2e724348dfa3231c38d3f91b76

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1                   

🏃 View run righteous-elk-216 at: http://127.0.0.1:5000/#/experiments/1/runs/0a906d2dfcb04f14be774537dcb493a3

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1                   

🏃 View run grandiose-cow-50 at: http://127.0.0.1:5000/#/experiments/1/runs/28012ae8ba16484e8f1cfffd43b23fa7

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1                   

🏃 View run funny-lynx-514 at: http://127.0.0.1:5000/#/experiments/1/runs/986cfec661f947fbbf45e20717f1f5bd

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1                   

🏃 View run classy-gull-681 at: http://127.0.0.1:5000/#/experiments/1/runs/b8c35956c75d4bb6a876bb6f1f5bc1ae

🧪 V

In [55]:
client = MlflowClient("http://127.0.0.1:5000")

exp = client.search_experiments()
for e in exp : 
    print(e.experiment_id, e.name)

1 random-forest-hyperopt
0 Default


In [ ]:
runs = client.search_runs(experiment_ids=[1])
for r in runs : 
    print(r.info.run_id, r.info.run_name, f"rmse : {r.data.metrics["rmse"]}")

2c5dd0723cd7466a8f9952ce39c11a0e hilarious-perch-761 rmse : 5.355041749098928
5263b4afde39494798ffd835c786dde0 casual-fly-512 rmse : 5.37159585587677
26a5987bcef646ff889e05df2dede7a3 calm-mule-46 rmse : 5.538069949003994
cde1b59db711461080f57b8d37f5823c grandiose-fox-657 rmse : 5.484068016681343
5b112e94f9d5497694b82da3f762218e masked-squid-68 rmse : 5.335419588556921
624f1c87640f4dfe8dee42aefb9463e6 gaudy-bat-541 rmse : 5.3633599989832135
e2c8025db073465e9f588e24c814be5d languid-gull-216 rmse : 5.410518705423985
81522ebd3e484a35a3a20b67290508bd bittersweet-grub-523 rmse : 5.4646431328121015
ccd5693e016c4bfea1ddf518a80d7aa4 dashing-shark-947 rmse : 5.363707729018604
8d7866beea17435c81b43032b1469fcf spiffy-hen-112 rmse : 5.35469507253029
b8c35956c75d4bb6a876bb6f1f5bc1ae classy-gull-681 rmse : 5.4431408201543805
986cfec661f947fbbf45e20717f1f5bd funny-lynx-514 rmse : 5.357490752366866
28012ae8ba16484e8f1cfffd43b23fa7 grandiose-cow-50 rmse : 5.617650247001616
0a906d2dfcb04f14be774537dcb493

In [60]:
best_run = min(runs, key=lambda r: r.data.metrics["rmse"])
print(best_run.info.run_id, best_run.info.run_name, best_run.data.metrics["rmse"])

5b112e94f9d5497694b82da3f762218e masked-squid-68 5.335419588556921


In [62]:
!python register_model.py

2026/11/01 22:44:20 INFO mlflow.tracking.fluent: Experiment with name 'random-forest-best-models' does not exist. Creating a new experiment.
2026/11/01 22:44:34 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/11/01 22:44:38 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
🏃 View run intrigued-chimp-557 at: http://127.0.0.1:5000/#/experiments/2/runs/d762ed109b42422ca1361588ac3dbdea
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
2026/11/01 22:44:59 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format req

In [ ]:
# 5.567